In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf, pacf

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
from IPython.display import display, Markdown

# Semana 8B — Autocorrelación y Estacionariedad (Laboratorio)

**Objetivo:** Diagnosticar la estructura temporal del dataset ClimaLab
usando ACF/PACF, evaluar estacionariedad y generar un inventario
de calidad de datos (gaps y registros sospechosos) para los bloques
4 y 5.

**Ejercicios:**

1. ACF y PACF de `tdb` horaria (ciclo diario)
2. ACF de `tdb` diaria (ciclo anual)
3. Estacionariedad en la práctica
4. Detección de gaps
5. Control de calidad

In [ ]:
df = pd.read_parquet("data/ClimaLab_2023-05-31_2025-06-20.parquet")

In [ ]:
tdb_horaria = df["tdb"].resample("1h").mean().dropna()
tdb_diaria = df["tdb"].resample("1D").mean().dropna()
ws_diaria = df["ws"].resample("1D").mean().dropna()

---
## 1. ACF y PACF de `tdb` horaria

Calculamos hasta lag = 72 h (3 días) para revelar:
- **Persistencia:** lags cortos con ACF alta
- **Ciclo diario:** picos cada 24 h

In [ ]:
_nlags = 72
_acf_vals = acf(tdb_horaria.values, nlags=_nlags, fft=True)
_pacf_vals = pacf(tdb_horaria.values, nlags=_nlags)
_ci = 1.96 / np.sqrt(len(tdb_horaria))

fig1, (ax1a, ax1b) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

# ACF
ax1a.bar(range(len(_acf_vals)), _acf_vals, color="steelblue",
         edgecolor="white", width=0.8)
ax1a.axhline(0, color="black", lw=0.5)
ax1a.axhline(_ci, color="crimson", ls="--", lw=1, alpha=0.6)
ax1a.axhline(-_ci, color="crimson", ls="--", lw=1, alpha=0.6)
# Marcar lags 24 y 48
for lag in [24, 48, 72]:
    if lag < len(_acf_vals):
        ax1a.axvline(lag, color="seagreen", ls=":", lw=1, alpha=0.5)
        ax1a.text(lag, _acf_vals[lag] + 0.02, f"lag {lag}",
                  fontsize=8, ha="center", color="seagreen")
ax1a.set_ylabel("ACF")
ax1a.set_title("ACF de tdb horaria (hasta 72 h)")

# PACF
ax1b.bar(range(len(_pacf_vals)), _pacf_vals, color="darkorange",
         edgecolor="white", width=0.8)
ax1b.axhline(0, color="black", lw=0.5)
ax1b.axhline(_ci, color="crimson", ls="--", lw=1, alpha=0.6)
ax1b.axhline(-_ci, color="crimson", ls="--", lw=1, alpha=0.6)
for lag in [1, 24]:
    ax1b.axvline(lag, color="seagreen", ls=":", lw=1, alpha=0.5)
ax1b.set_xlabel("Lag (horas)")
ax1b.set_ylabel("PACF")
ax1b.set_title("PACF de tdb horaria")

plt.tight_layout()
plt.show()

> **Observaciones:**
> - **ACF:** Decae lentamente (persistencia térmica) con picos
>   cada 24 h (ciclo diurno).  La temperatura de las 3 PM está
>   fuertemente correlacionada con la de las 2 PM, y también
>   con la de las 3 PM del día anterior.
> - **PACF:** Pico dominante en lag 1 (el dato inmediatamente anterior
>   es el más informativo) y pico secundario en lag 24 (el ciclo diario
>   aporta información independiente).
> - Esto anticipa que un modelo AR para temperatura horaria necesitará
>   componentes en lag 1 y lag 24 → motivación para SARIMA (semana 13).

---
## 2. ACF de `tdb` y `ws` diarias (largo plazo)

Calculamos hasta lag = 400 días para buscar el ciclo anual
(pico en lag ≈ 365).

In [ ]:
_nlags = 400
_acf_tdb = acf(tdb_diaria.values, nlags=_nlags, fft=True)
_acf_ws = acf(ws_diaria.values, nlags=_nlags, fft=True)

fig2, (ax2a, ax2b) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

_ci_tdb = 1.96 / np.sqrt(len(tdb_diaria))
_ci_ws = 1.96 / np.sqrt(len(ws_diaria))

ax2a.plot(range(len(_acf_tdb)), _acf_tdb, color="steelblue", lw=0.8)
ax2a.axhline(0, color="black", lw=0.5)
ax2a.axhline(_ci_tdb, color="crimson", ls="--", lw=1, alpha=0.4)
ax2a.axhline(-_ci_tdb, color="crimson", ls="--", lw=1, alpha=0.4)
ax2a.axvline(365, color="seagreen", ls=":", lw=1.5,
             label="lag = 365 (ciclo anual)")
ax2a.set_ylabel("ACF")
ax2a.set_title("ACF de tdb diaria")
ax2a.legend()

ax2b.plot(range(len(_acf_ws)), _acf_ws, color="darkorange", lw=0.8)
ax2b.axhline(0, color="black", lw=0.5)
ax2b.axhline(_ci_ws, color="crimson", ls="--", lw=1, alpha=0.4)
ax2b.axhline(-_ci_ws, color="crimson", ls="--", lw=1, alpha=0.4)
ax2b.axvline(365, color="seagreen", ls=":", lw=1.5,
             label="lag = 365")
ax2b.set_xlabel("Lag (días)")
ax2b.set_ylabel("ACF")
ax2b.set_title("ACF de ws diaria")
ax2b.legend()

plt.tight_layout()
plt.show()

> **Comparación:**
> - **`tdb`:** ACF decae muy lentamente y muestra un pico
>   alrededor de lag 365 (ciclo anual).  La temperatura tiene
>   *mucha* memoria temporal.
> - **`ws`:** ACF cae a ~0 mucho más rápido.  El viento es más
>   "ruidoso" — su valor de hoy dice poco sobre el de hace 30 días.
>   El ciclo anual del viento es débil o ausente.

---
## 3. Estacionariedad en la práctica

Dividimos `tdb` diaria en ventanas de 3 meses y verificamos si
la media y la varianza son estables.  Luego diferenciamos y
repetimos el análisis.

In [ ]:
_y = tdb_diaria.values
_idx = tdb_diaria.index
_dy = np.diff(_y)
_idx_diff = _idx[1:]

# Ventanas de 90 días
_ventana = 90

fig3, axes3 = plt.subplots(2, 2, figsize=(13, 8))

for col, (serie, indice, titulo, color) in enumerate([
    (_y, _idx, "tdb diaria (original)", "steelblue"),
    (_dy, _idx_diff, "Δtdb (diferenciada)", "seagreen"),
]):
    # Serie
    axes3[0, col].plot(indice, serie, color=color, lw=0.5)
    axes3[0, col].set_title(titulo)
    axes3[0, col].set_ylabel("°C" if col == 0 else "Δ°C")

    # Media y DE por ventanas
    _medias = []
    _des = []
    _centros = []
    for start in range(0, len(serie) - _ventana, _ventana):
        _bloque = serie[start : start + _ventana]
        _medias.append(np.nanmean(_bloque))
        _des.append(np.nanstd(_bloque))
        _centros.append(indice[start + _ventana // 2])

    ax = axes3[1, col]
    ax.plot(_centros, _medias, "o-", color=color, markersize=5,
            lw=1.5, label="Media")
    ax_twin = ax.twinx()
    ax_twin.plot(_centros, _des, "s--", color="gray", markersize=5,
                 lw=1.5, alpha=0.7, label="DE")
    ax.set_ylabel("Media", color=color)
    ax_twin.set_ylabel("DE", color="gray")
    ax.set_xlabel("Fecha")
    ax.set_title("Media y DE por ventanas de 90 días")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

> **Resultado:**
> - **Original:** La media varía fuertemente entre ~18 °C (invierno) y
>   ~29 °C (verano) → **no estacionaria en media**.
> - **Diferenciada (Δtdb):** La media es ~0 y la DE es más estable
>   → mucho más cercana a estacionaria.
>
> La diferenciación eliminó la tendencia/estacionalidad de baja frecuencia.
> Pero la estacionalidad *anual* no desaparece completamente con d=1 —
> para eso se necesita diferenciación estacional (SARIMA, semana 13).

---
## 4. Detección de gaps

Calculamos la diferencia entre timestamps consecutivos.
Donde `diff > 1 minuto`, hay un gap (datos faltantes).

In [ ]:
_diffs = df.index.to_series().diff()
_un_minuto = pd.Timedelta(minutes=1)
_gaps = _diffs[_diffs > _un_minuto].copy()
gaps_df = pd.DataFrame({
    "inicio": _gaps.index - _gaps.values + _un_minuto,
    "fin": _gaps.index,
    "duración": _gaps.values,
    "registros_faltantes": (_gaps.values / _un_minuto).astype(int) - 1,
}).sort_values("duración", ascending=False).reset_index(drop=True)

n_gaps = len(gaps_df)
total_faltantes = gaps_df["registros_faltantes"].sum()

display(Markdown(f"""
### Resumen de gaps

- **Total de gaps detectados:** {n_gaps}
- **Total de registros faltantes:** {total_faltantes:,}
- **Gap más largo:** {gaps_df["duración"].iloc[0] if n_gaps > 0 else "N/A"}

### Top 10 gaps más largos
"""))

In [ ]:
display(gaps_df.head(10))

In [ ]:
fig4, ax4 = plt.subplots(figsize=(13, 4))

_tdb = df["tdb"].resample("1h").mean()
ax4.plot(_tdb.index, _tdb.values, color="steelblue", lw=0.3, alpha=0.7)

# Marcar gaps como barras rojas
for _, row in gaps_df.iterrows():
    if row["duración"] > pd.Timedelta(hours=1):
        ax4.axvspan(row["inicio"], row["fin"],
                    color="crimson", alpha=0.3)

ax4.set_xlabel("Fecha")
ax4.set_ylabel("tdb (°C)")
ax4.set_title("Serie tdb horaria — gaps marcados en rojo (> 1 hora)")
plt.tight_layout()
plt.show()

> Este inventario de gaps se usará en el **bloque 5 (imputación)**
> para decidir qué segmentos necesitan ser rellenados y con qué método
> según la duración del gap.

---
## 5. Control de calidad

Verificamos la integridad de los datos aplicando reglas físicas
y comprobaciones básicas de timestamps.

### 5.1 Timestamps

In [ ]:
_n_dup = df.index.duplicated().sum()
_monotonic = df.index.is_monotonic_increasing
_freq_mode = df.index.to_series().diff().dropna().mode().iloc[0]

display(Markdown(f"""
| Verificación | Resultado |
|:---|:---|
| Timestamps duplicados | {_n_dup} |
| Orden cronológico | {"✓ Sí" if _monotonic else "✗ No"} |
| Frecuencia más común | {_freq_mode} |
"""))

### 5.2 Reglas físicas

Detectamos registros que violan restricciones del dominio:

In [ ]:
_reglas = {
    "rh < 0 (humedad negativa)": (df["rh"] < 0).sum(),
    "p_atm < 500 hPa (sensor defectuoso)": (df["p_atm"] < 500).sum(),
    "tdb > 45 °C (extremo para la región)": (df["tdb"] > 45).sum(),
    "tdb < -5 °C (extremo para la región)": (df["tdb"] < -5).sum(),
    "ghi > 0 cuando solar_alt < -5° (noche)": (
        (df["ghi"] > 5) & (df["solar_altitude"] < -5)
    ).sum(),
}

_rows = ""
for regla, conteo in _reglas.items():
    _icono = "⚠" if conteo > 0 else "✓"
    _rows += f"| {regla} | {conteo:,} | {_icono} |\n"

display(Markdown(f"""
| Regla | # Violaciones | Estado |
|:---|---:|:---|
{_rows}

> Los registros con violaciones son candidatos a **outliers por
> reglas físicas** — los revisaremos en la **semana 9** (detección
> de anomalías).
"""))

### 5.3 Visualización de registros sospechosos

In [ ]:
fig5, axes5 = plt.subplots(1, 3, figsize=(14, 4))

# rh negativa
_rh = df["rh"]
_mask_rh = _rh < 0
axes5[0].plot(_rh.index, _rh.values, "steelblue", lw=0.1, alpha=0.3)
axes5[0].scatter(_rh.index[_mask_rh], _rh.values[_mask_rh],
                 c="crimson", s=5, zorder=5, label=f"rh < 0 ({_mask_rh.sum()})")
axes5[0].set_ylabel("rh (%)")
axes5[0].set_title("Humedad relativa")
axes5[0].legend(fontsize=8)

# p_atm extrema
_patm = df["p_atm"]
_mask_patm = _patm < 500
axes5[1].plot(_patm.index, _patm.values, "steelblue", lw=0.1, alpha=0.3)
axes5[1].scatter(_patm.index[_mask_patm], _patm.values[_mask_patm],
                 c="crimson", s=5, zorder=5,
                 label=f"p_atm < 500 ({_mask_patm.sum()})")
axes5[1].set_ylabel("p_atm (hPa)")
axes5[1].set_title("Presión atmosférica")
axes5[1].legend(fontsize=8)

# ghi de noche
_ghi_noche = (df["ghi"] > 5) & (df["solar_altitude"] < -5)
_ghi = df["ghi"]
axes5[2].scatter(
    df.index[_ghi_noche],
    _ghi.values[_ghi_noche],
    c="crimson", s=3, alpha=0.5,
    label=f"ghi > 5 de noche ({_ghi_noche.sum()})",
)
axes5[2].set_ylabel("ghi (W/m²)")
axes5[2].set_title("Irradiancia nocturna anómala")
axes5[2].legend(fontsize=8)

for ax in axes5:
    ax.tick_params(axis="x", rotation=30)

plt.suptitle("Registros sospechosos por reglas físicas", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## Resumen de la sesión

| Ejercicio | Hallazgo principal |
|:---|:---|
| **ACF/PACF horaria** | Persistencia fuerte + picos cada 24 h; PACF indica lags 1 y 24 como los más informativos |
| **ACF diaria** | `tdb` tiene ciclo anual claro; `ws` tiene poca memoria a largo plazo |
| **Estacionariedad** | `tdb` diaria es no estacionaria; Δtdb es más estable pero conserva estacionalidad anual |
| **Gaps** | Inventario completo de períodos sin datos — input para el bloque de imputación |
| **QC** | Humedad negativa, presión extrema e irradiancia nocturna detectados como anomalías candidatas |

**Próxima semana (9):** Detección formal de anomalías con métodos
estadísticos (z-score, MAD, IQR).